<a href="https://colab.research.google.com/github/susuelen153-design/Projeto-Python/blob/main/PYTHON_SOMPO_CHALLANGE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Equipe Perceptron | Sala 1TIAPZ1
Matéria: Computational Thinking with Python | Professor: Luciana
Sprint 1 | Análise de Risco Operacional com Python

Giovanni Henrique Pereira Hessel (RM 570574)

Suellen Pereira da Silva (RM 573862)

Arthur Zeferino (RM 570858)

Israel Carneiro de Toledo (RM 573854)

Luan Gonçalves de Freitas (RM 571935)

## Instruções

Projeto: Análise de Risco Operacional com Python, Sompo Seguros

Desenvolver uma solução em Python para análise de risco em equipamentos agrícolas, identificando fatores ambientais e operacionais que aumentam a probabilidade de dano ou perda, com base em dados históricos e variáveis de contexto. A solução permite classificação de risco, geração de alertas preventivos e apoio à tomada de decisão, alinhada ao desafio SOMPO.


### O Problema

A Sompo Seguros opera no seguro rural com produtos como Penhor Rural, Benfeitorias, RD Equipamentos, Agrícola e Transportes. Em 2024, o segmento rural registrou R$ 461 milhões em prêmios e R$ 338 milhões em sinistros, atingindo 73% de sinistralidade, bem acima do sustentável.

Sinistros cresceram 22,5% de 2023 para 2024 (1,3 bi para R$ 1,6 bi), com sinistralidade geral de 68,8%. Indenizações por perdas em máquinas agrícolas somaram R$ 842,3 milhões em 2024 (+25,4% YoY, fonte CNseg/Globo Rural).

O gargalo: a seguradora não enxerga o ativo entre a apólice e o sinistro. Dados operacionais fragmentados entre fabricante, corretor e seguradora. O seguro é 100% reativo.

### Solução: Plataforma Perceptron

Solução orientada a dados em três camadas:

- Ingestão: telemetria nativa via APIs dos fabricantes (18 integrados via Addiante), APIs de clima e geografia, histórico de sinistros e manutenção
- Motor Analítico: score de risco explicável por equipamento, região e tipo de operação; ranking de ativos críticos; detecção de anomalias
- Ação Preventiva alertas antes do evento (geofencing, clima, manutenção, código de erro); Caixa-Preta com snapshot pré-sinistro; Selo Sompo com certificação telemétrica

Riscos identificados: colisão e tombamento, operação próxima de água, manutenção vencida, evento climático extremo, desvio de uso, comportamento de risco do operador, fraude.

Fatores operacionais: telemetria (GPS, velocidade, DTC, alarmas), regime de operação (campo, transporte, proximidade de água), histórico de manutenção, comportamento do operador, conformidade.

Fatores ambientais: clima (chuva, inundação, ventania, calor, geada), geografia (terreno, altitude, corpos d'água), sazonalidade (safra vs. entressafra).


### Funcionalidades Esperadas

| # | Funcionalidade | Descrição |
|---|---|---|
| 1 | **Ingestão e estruturação** | Leitura de CSV com pandas, identificação de tipos e estrutura da base |
| 2 | **Preparação e limpeza** | Tratamento de nulos, padronização de categorias, correção de inconsistências |
| 3 | **Feature Engineering** | Variáveis de risco operacional, ambiental, frequência de incidentes e zona crítica |
| 4 | **Score de Risco** | Score ponderado por equipamento com classificação Baixo/Médio/Alto |
| 5 | **Alertas preventivos** | Operação próxima à água, histórico de incidentes, clima adverso, manutenção pendente |
| 6 | **Relatórios** | Top equipamentos por risco, ranking por região e operação, fatores de influência |
| 7 | **Menu interativo** | Interface via terminal com opções numeradas para todas as funcionalidades |

### Requisitos Técnicos

| Requisito | Pontos | Uso no código |
|---|---|---|
| Listas e dicionários | 2,5 | Listas (alertas, colunas) e dicionários (pesos, mapeamentos, opções de menu) |
| Funções para modularização | 2,5 | 12 funções: limpeza, cálculo de risco, alertas, relatórios e menu |
| Estruturas condicionais (if/else) | 2,5 | Classificação de risco, regras de alerta, zona crítica e menu |
| Pandas para manipulação de dados | 2,5 | Leitura CSV, fillna, groupby/agg, sort_values, apply, corr, iterrows |

### Referências

- Demonstrativo Financeiro Sompo 2024
- Release Sompo 2024: Seguro Rural
- Farsul/S.O.S Agro RS: Enchentes RS 2024
- CNseg/Globo Rural: Indenizações máquinas agrícolas 2024
- Reunião Fábio Leite, CEO Addiante (17/04/2026)
- Apresentação Sprint 1 à Sompo (06/05/2026)

In [ ]:
# 1. INGESTÃO E ESTRUTURAÇÃO DOS DADOS
# Leitura do CSV com informações operacionais, ambientais e histórico de incidentes de equipamentos agrícolas.

import pandas as pd
from google.colab import files

url = 'https://raw.githubusercontent.com/susuelen153-design/Projeto-Python/refs/heads/main/base_sompo.csv'

# O pandas consegue ler a URL diretamente
df = pd.read_csv(url)

df.info()

In [ ]:
# EXPLORAÇÃO INICIAL
# Tipos de dados, estrutura, estatísticas e categorias disponíveis.

print("=== Primeiros registros da base ===")
print(df.head())

print("\n=== Estrutura da base ===")
print(df.info())

print("\n=== Estatísticas gerais ===")
print(df.describe())

colunas_texto = ['equipamento', 'regiao', 'operacao', 'condicao']

print("\n=== Categorias disponíveis ===")
for coluna in colunas_texto:
    print(f"\n{coluna}: {df[coluna].unique()}")

In [ ]:
# 2. PREPARAÇÃO E LIMPEZA DOS DADOS
# Trata nulos (mediana numérica), padroniza texto (Title Case), corrige variações de escrita e garante tipos binários.
# Usa: listas (colunas_numericas, colunas_texto), dicionário (correcoes), condicionais (if/else) e pandas.

def limpar_dados(df):
    df = df.copy()

    nulos = df.isnull().sum()
    print("=== Valores nulos por coluna ===")
    print(nulos)

    # Nulos numéricos: preenche com mediana
    colunas_numericas = ['incidentes', 'idade_anos', 'dias_sem_manutencao']
    for col in colunas_numericas:
        if df[col].isnull().any():
            mediana = df[col].median()
            df[col] = df[col].fillna(mediana)
            print(f"  → Nulos em '{col}' preenchidos com mediana ({mediana})")

    # Texto: strip + Title Case
    colunas_texto = ['equipamento', 'regiao', 'operacao', 'condicao']
    for col in colunas_texto:
        df[col] = df[col].astype(str).str.strip().str.title()

    # Corrige variações de escrita da operação próxima à água
    correcoes = {
        'Proximo_Agua': 'Proximo_Agua',
        'Proximo A Agua': 'Proximo_Agua',
        'Próximo À Água': 'Proximo_Agua',
    }
    df['operacao'] = df['operacao'].replace(correcoes)

    # Colunas binárias: garante 0 ou 1
    for col in ['proximo_agua', 'dano']:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

    print("\n✔ Limpeza concluída! Shape final:", df.shape)
    return df

df = limpar_dados(df)
df.head()

In [ ]:
# 3. FEATURE ENGINEERING — SCORE DE RISCO
# Cria 4 variáveis analíticas (risco operacional, ambiental, frequência de incidentes, zona crítica) e gera o score ponderado final.
# Usa: dicionários (pesos, risco_por_operacao, mapa_condicao), funções modularizadas, condicionais (if/elif/else) e pandas.

pesos = {
    'risco_operacional': 0.30,
    'risco_ambiental':   0.25,
    'freq_incidentes':   0.30,
    'zona_critica':      0.15,
}

risco_por_operacao = {
    'Proximo_Agua': 90,
    'Colheita':     65,
    'Transporte':   55,
    'Campo':        35,
}


def calcular_risco_operacional(tipo_operacao):
    """Score de risco baseado no tipo de operação."""
    return risco_por_operacao.get(tipo_operacao, 50)


def calcular_risco_ambiental(row):
    """Combina condição climática e proximidade à água."""
    score = 0

    mapa_condicao = {
        'Tempestade': 80,
        'Chuva':      45,
        'Seca':       35,
        'Normal':     10,
    }
    score += mapa_condicao.get(row['condicao'], 20)

    if row['proximo_agua'] == 1:
        score += 20

    return min(score, 100)


def calcular_zona_critica(row):
    """Pondera região, idade do equipamento e tempo sem manutenção."""
    pontos = 0

    if row['regiao'] in ['Norte', 'Nordeste']:
        pontos += 30
    elif row['regiao'] == 'Centro-Oeste':
        pontos += 20
    else:
        pontos += 10

    if row['idade_anos'] > 10 and row['dias_sem_manutencao'] > 120:
        pontos += 40
    elif row['idade_anos'] > 7 and row['dias_sem_manutencao'] > 60:
        pontos += 20

    return min(pontos, 100)


def criar_features(df):
    """Cria variáveis analíticas e score final ponderado."""

    df = df.copy()

    df['risco_operacional'] = df['operacao'].apply(calcular_risco_operacional)
    df['risco_ambiental'] = df.apply(calcular_risco_ambiental, axis=1)

    maximo = df['incidentes'].max()
    df['freq_incidentes'] = (df['incidentes'] / maximo * 100).round(1) if maximo > 0 else 0

    df['zona_critica'] = df.apply(calcular_zona_critica, axis=1)

    # Score ponderado: operacional (30%) + incidentes (30%) + ambiental (25%) + zona (15%)
    df['score_risco'] = (
        df['risco_operacional'] * pesos['risco_operacional'] +
        df['risco_ambiental']   * pesos['risco_ambiental']   +
        df['freq_incidentes']   * pesos['freq_incidentes']   +
        df['zona_critica']      * pesos['zona_critica']
    ).round(1)

    def classificar(score):
        if score <= 33:
            return 'Baixo'
        elif score <= 66:
            return 'Médio'
        else:
            return 'Alto'

    df['classificacao'] = df['score_risco'].apply(classificar)

    print("✔ Features criadas com sucesso!")
    print(df[['equipamento', 'regiao', 'risco_operacional',
              'risco_ambiental', 'freq_incidentes',
              'zona_critica', 'score_risco', 'classificacao']].head(10))
    return df

df = criar_features(df)

In [ ]:
# 4. GERAÇÃO DE ALERTAS
# Varre a base e identifica 4 condições críticas: operação próxima à água com score alto, histórico pesado de incidentes, tempestade ativa e manutenção crítica pendente.
# Usa: lista (alertas), dicionários (cada alerta), condicionais e pandas.

def gerar_alertas(df):
    alertas = []

    for _, row in df.iterrows():
        eq  = row['equipamento']
        reg = row['regiao']
        sc  = row['score_risco']
        cl  = row['classificacao']

        # Operação próxima à água com score elevado
        if row['proximo_agua'] == 1 and sc > 50:
            alertas.append({
                'equipamento':  eq,
                'regiao':       reg,
                'tipo_alerta':  'Risco Elevado — Operação Próxima à Água',
                'score':        sc,
                'classificacao': cl,
                'descricao':    'Risco de corrosão, curto elétrico e sinistro total.',
            })

        # Histórico pesado de incidentes (≥4)
        if row['incidentes'] >= 4:
            alertas.append({
                'equipamento':  eq,
                'regiao':       reg,
                'tipo_alerta':  'Alto Histórico de Incidentes',
                'score':        sc,
                'classificacao': cl,
                'descricao':    f"{int(row['incidentes'])} incidentes registrados. Revisão completa necessária.",
            })

        # Tempestade ativa: suspender operação
        if row['condicao'] == 'Tempestade':
            alertas.append({
                'equipamento':  eq,
                'regiao':       reg,
                'tipo_alerta':  'Condições Climáticas Críticas',
                'score':        sc,
                'classificacao': cl,
                'descricao':    'Tempestade ativa. Suspender operação imediatamente.',
            })

        # Equipamento velho com manutenção atrasada
        if row['idade_anos'] > 10 and row['dias_sem_manutencao'] > 150:
            alertas.append({
                'equipamento':  eq,
                'regiao':       reg,
                'tipo_alerta':  'Manutenção Crítica Pendente',
                'score':        sc,
                'classificacao': cl,
                'descricao':    f"{int(row['idade_anos'])} anos / {int(row['dias_sem_manutencao'])} dias sem revisão.",
            })

    print(f"✔ {len(alertas)} alertas gerados!")
    return alertas

alertas = gerar_alertas(df)

In [ ]:
# 5. RELATÓRIOS
# 4 funções de relatório: top equipamentos por risco, ranking por região, ranking por operação e correlação dos fatores de influência.
# Usa: pandas (groupby, agg, sort_values, corr) e formatação.

def relatorio_top_risco(df, n=10):
    """Lista os N equipamentos com maior score de risco."""
    print(f"\n{'='*55}")
    print(f" TOP {n} — EQUIPAMENTOS COM MAIOR RISCO")
    print(f"{'='*55}")
    top = (df[['equipamento', 'regiao', 'operacao', 'score_risco', 'classificacao']]
           .sort_values('score_risco', ascending=False)
           .head(n)
           .reset_index(drop=True))
    print(top.to_string(index=False))


def relatorio_por_regiao(df):
    """Ranking de risco médio por região."""
    print(f"\n{'='*55}")
    print(" RANKING DE RISCO POR REGIÃO")
    print(f"{'='*55}")
    ranking = (df.groupby('regiao')
                 .agg(score_medio=('score_risco', 'mean'),
                      equipamentos=('id', 'count'),
                      incidentes=('incidentes', 'sum'),
                      danos=('dano', 'sum'))
                 .sort_values('score_medio', ascending=False)
                 .round(1))
    print(ranking.to_string())


def relatorio_por_operacao(df):
    """Ranking de risco médio por tipo de operação."""
    print(f"\n{'='*55}")
    print(" RANKING DE RISCO POR TIPO DE OPERAÇÃO")
    print(f"{'='*55}")
    ranking = (df.groupby('operacao')
                 .agg(score_medio=('score_risco', 'mean'),
                      equipamentos=('id', 'count'),
                      taxa_dano=('dano', 'mean'))
                 .sort_values('score_medio', ascending=False)
                 .round(2))
    ranking['taxa_dano'] = (ranking['taxa_dano'] * 100).round(1).astype(str) + '%'
    print(ranking.to_string())


def relatorio_fatores(df):
    """Correlação de cada feature com o score final."""
    print(f"\n{'='*55}")
    print(" FATORES QUE MAIS INFLUENCIAM O RISCO")
    print(f"{'='*55}")
    features = ['risco_operacional', 'risco_ambiental', 'freq_incidentes', 'zona_critica']
    for feat in features:
        corr = df[feat].corr(df['score_risco'])
        barra = '█' * int(abs(corr) * 30)
        print(f"  {feat:25} | {corr:.3f}  {barra}")

    print(f"\n  Distribuição por classificação:")
    for cls in ['Alto', 'Médio', 'Baixo']:
        qtd = len(df[df['classificacao'] == cls])
        pct = qtd / len(df) * 100
        print(f"    {cls:8}: {qtd:3} equipamentos ({pct:.1f}%)")

In [ ]:







# 6. MENU INTERATIVO
# Interface via terminal com 9 opções. Usa dicionário (opcoes), condicionais (if/elif/else), funções e laço while.

import pandas as pd


def exibir_alertas(alertas):
    """Exibe todos os alertas preventivos formatados."""
    print(f"\n{'='*40}")
    print(" ALERTAS PREVENTIVOS")
    print(f"{'='*40}")

    if not alertas:
        print(" > Tudo em ordem. Nenhum alerta crítico.")
        return

    for a in alertas:
        print(f"\n[!] {a['tipo_alerta']}")
        print(f"    Equipamento: {a['equipamento']} ({a['regiao']})")
        print(f"    Status: {a['score']} - {a['classificacao']}")
        print(f"    Nota: {a['descricao']}")


def main():
    dados = None
    alertas = None

    opcoes = {
        '1': 'Carregar e limpar os dados',
        '2': 'Processar indicadores de risco',
        '3': 'Relatório: Top 10 em Risco',
        '4': 'Relatório: Risco por Região',
        '5': 'Relatório: Risco por Operação',
        '6': 'Relatório: Fatores de Influência',
        '7': 'Exibir Alertas Preventivos',
        '8': 'Executar Análise Completa',
        '0': 'Sair',
    }

    while True:
        print(f"\n{'='*45}")
        print("   SISTEMA DE ANÁLISE DE RISCO")
        print(f"{'='*45}")

        for num, desc in opcoes.items():
            print(f" {num}. {desc}")

        escolha = input("\n> Selecione uma opção: ").strip()

        if escolha == '0':
            print("Encerrando sistema...")
            break

        if escolha == '1':
            dados = limpar_dados(df)
            alertas = None
            print("[OK] Dados carregados.")

        elif escolha == '2':
            if dados is not None:
                dados = criar_features(dados)
                alertas = gerar_alertas(dados)
                print("[OK] Indicadores de risco gerados.")
            else:
                print("[!] Erro: Carregue os dados (Opção 1) primeiro.")

        elif escolha in ['3', '4', '5', '6']:
            if dados is not None and 'score_risco' in dados.columns:
                if escolha == '3': relatorio_top_risco(dados)
                if escolha == '4': relatorio_por_regiao(dados)
                if escolha == '5': relatorio_por_operacao(dados)
                if escolha == '6': relatorio_fatores(dados)
            else:
                print("[!] Erro: Processe os riscos (Opção 2) primeiro.")

        elif escolha == '7':
            if alertas:
                exibir_alertas(alertas)
            else:
                print("[!] Nenhum alerta gerado ou dados não processados.")

        elif escolha == '8':
            print("\nIniciando processamento total...")
            dados = limpar_dados(df)
            dados = criar_features(dados)
            alertas = gerar_alertas(dados)

            relatorio_top_risco(dados)
            relatorio_por_regiao(dados)
            relatorio_por_operacao(dados)
            relatorio_fatores(dados)
            exibir_alertas(alertas)
            print("\n[OK] Análise finalizada com sucesso.")

        else:
            print("[!] Opção inválida.")


if __name__ == "__main__":
    main()


   SISTEMA DE ANÁLISE DE RISCO
 1. Carregar e limpar os dados
 2. Processar indicadores de risco
 3. Relatório: Top 10 em Risco
 4. Relatório: Risco por Região
 5. Relatório: Risco por Operação
 6. Relatório: Fatores de Influência
 7. Exibir Alertas Preventivos
 8. Executar Análise Completa
 0. Sair

> Selecione uma opção: 1
=== Valores nulos por coluna ===
id                     0
equipamento            0
regiao                 0
operacao               0
condicao               0
incidentes             0
idade_anos             0
dias_sem_manutencao    0
proximo_agua           0
dano                   0
risco_operacional      0
risco_ambiental        0
freq_incidentes        0
zona_critica           0
score_risco            0
classificacao          0
dtype: int64

✔ Limpeza concluída! Shape final: (50, 16)
[OK] Dados carregados.

   SISTEMA DE ANÁLISE DE RISCO
 1. Carregar e limpar os dados
 2. Processar indicadores de risco
 3. Relatório: Top 10 em Risco
 4. Relatório: Risco por Regiã